# 02 — Genre Analysis

Analyse des genres de jeux sur Steam : popularité, performance (ratings), et rentabilité estimée.

**Objectif métier** : Identifier les genres porteurs et les opportunités de niche pour un nouveau jeu.

**Caveat important** : `estimated_revenue` est un proxy approximatif (price × avg_owners). Les owners ne sont pas tous des acheteurs payants, et les jeux gratuits sont exclus de l'analyse de rentabilité.

In [0]:
from functools import reduce

from pyspark.sql import DataFrame, functions as F

df = spark.table("steam_clean")
print(f"Total games: {df.count()}")

df_genres = (df
    .filter(F.col("genre").isNotNull())
    .withColumn("genre_single", F.explode(F.col("genre")))
)
print(f"Total game-genre combinations: {df_genres.count()}")

Total games: 55690
Total game-genre combinations: 157110


## Q7 — Which genres are most represented?

Analysis of genre distribution across all Steam games.

In [0]:
df_genre_counts = (df_genres
    .groupBy("genre_single")
    .agg(F.count("*").alias("game_count"))
    .orderBy(F.desc("game_count"))
    .limit(15)
)
display(df_genre_counts)

genre_single,game_count
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Databricks visualization. Run in Databricks to view.

## Q8 — What is the average positive/negative ratio by genre?

Average `rating_ratio` and `pos_neg_ratio` per genre (minimum 300 games, only games with reviews).

In [0]:
df_genre_ratings = (df_genres
    .filter(F.col("total_reviews") > 0)
    .groupBy("genre_single")
    .agg(
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.avg("pos_neg_ratio"), 2).alias("avg_pos_neg_ratio"),
        F.count("*").alias("game_count"),
        F.round(F.avg("total_reviews"), 0).alias("avg_reviews")
    )
    .filter(F.col("game_count") >= 300)
    .orderBy(F.desc("avg_rating"))
)
display(df_genre_ratings)

genre_single,avg_rating,avg_pos_neg_ratio,game_count,avg_reviews
Casual,0.743,6.05,21990,526.0
Indie,0.742,5.96,39558,930.0
Adventure,0.739,6.24,21397,1652.0
RPG,0.731,5.64,9520,2384.0
Action,0.73,5.38,23695,2724.0
Free to Play,0.722,5.29,3388,6790.0
Design & Illustration,0.721,5.39,405,1731.0
Strategy,0.719,4.78,10876,1452.0
Early Access,0.707,4.69,6128,860.0
Animation & Modeling,0.707,5.51,321,2234.0


Databricks visualization. Run in Databricks to view.

`pos_neg_ratio` coverage: 84.2% (8,797 games with zero negative reviews excluded). `rating_ratio` coverage: 99.7%.

## Q9 — Do publishers have favorite genres?

Cross-analysis: top 5 publishers and their preferred genres.

In [0]:
top_publishers = (df
    .filter(F.col("publisher").isNotNull())
    .groupBy("publisher")
    .agg(F.count("*").alias("total_games"))
    .orderBy(F.desc("total_games"))
    .limit(5)
    .select("publisher")
    .collect()
)
top_publisher_list = [row.publisher for row in top_publishers]
print(f"Top 5 publishers: {top_publisher_list}")

Top 5 publishers: ['Big Fish Games', '8floor', 'SEGA', 'Strategy First', 'Square Enix']


In [0]:
df_pub_genres = reduce(
    DataFrame.unionAll,
    [
        df_genres
        .filter(F.col("publisher") == p)
        .groupBy("publisher", "genre_single")
        .agg(F.count(F.lit(1)).alias("genre_count"))
        .orderBy(F.desc("genre_count"))
        .limit(3)
        for p in top_publisher_list
    ],
).orderBy("publisher", F.desc("genre_count"))

display(df_pub_genres)

publisher,genre_single,genre_count
8floor,Casual,202
8floor,Strategy,22
8floor,Simulation,10
Big Fish Games,Casual,418
Big Fish Games,Adventure,392
Big Fish Games,Simulation,7
SEGA,Action,80
SEGA,Adventure,33
SEGA,Strategy,32
Square Enix,Action,75


Databricks visualization. Run in Databricks to view.

In [0]:
top_genres = [row["genre_single"] for row in df_genre_counts.limit(7).collect()]

df_heatmap = (df_genres
    .filter(F.col("publisher").isin(top_publisher_list))
    .filter(F.col("genre_single").isin(top_genres))
    .groupBy("publisher", "genre_single")
    .agg(F.count(F.lit(1)).alias("count"))
)
display(df_heatmap)

publisher,genre_single,count
Big Fish Games,Casual,418
Strategy First,Action,28
Big Fish Games,Strategy,6
8floor,Adventure,8
Strategy First,RPG,5
8floor,Simulation,10
SEGA,Adventure,33
Square Enix,Simulation,7
Strategy First,Casual,25
Square Enix,Action,75


Databricks visualization. Run in Databricks to view.

## Q10 — Which genres are most profitable?

Average `estimated_revenue` per genre. Free games excluded to avoid bias.

In [0]:
df_genre_revenue = (df_genres
    .filter(~F.col("is_free"))
    .filter(F.col("estimated_revenue").isNotNull())
    .groupBy("genre_single")
    .agg(
        F.round(F.avg("estimated_revenue"), 0).alias("avg_revenue"),
        F.round(F.percentile_approx("estimated_revenue", 0.5), 0).alias("median_revenue"),
        F.round(F.sum("estimated_revenue"), 0).alias("total_revenue"),
        F.count("*").alias("paid_game_count"),
        F.round(F.avg("price_usd"), 2).alias("avg_price")
    )
    .filter(F.col("paid_game_count") >= 300)
    .orderBy(F.desc("avg_revenue"))
)
display(df_genre_revenue)

genre_single,avg_revenue,median_revenue,total_revenue,paid_game_count,avg_price
Massively Multiplayer,8644545.0,109900.0,5.93015775E9,686,10.64
RPG,3338224.0,109900.0,2.71731431E10,8140,10.59
Action,2854749.0,89900.0,5.87564541E10,20582,8.92
Strategy,2146819.0,99900.0,2.015004105E10,9386,9.76
Simulation,1986007.0,99900.0,1.876974955E10,9451,10.42
Adventure,1958344.0,99900.0,3.724573845E10,19019,9.02
Racing,1447121.0,79900.0,2.723481E9,1882,9.41
Sports,1398090.0,99900.0,3.14989735E9,2253,10.59
Early Access,1081780.0,99900.0,5.45866145E9,5046,10.65
Indie,930435.0,69900.0,3.23465771E10,34765,7.5


Databricks visualization. Run in Databricks to view.

In [0]:
df_revenue_rating = (df_genres
    .filter(~F.col("is_free"))
    .filter(F.col("total_reviews") > 0)
    .groupBy("genre_single")
    .agg(
        F.round(F.avg("estimated_revenue"), 0).alias("avg_revenue"),
        F.round(F.avg("rating_ratio"), 3).alias("avg_rating"),
        F.round(F.avg("pos_neg_ratio"), 2).alias("avg_pos_neg_ratio"),
        F.count("*").alias("count")
    )
    .filter(F.col("count") >= 100)
    .orderBy(F.desc("avg_revenue"))
    .limit(10)
)
display(df_revenue_rating)

genre_single,avg_revenue,avg_rating,avg_pos_neg_ratio,count
Massively Multiplayer,8644545.0,0.636,3.18,686
RPG,3343392.0,0.739,5.86,8127
Action,2862516.0,0.737,5.6,20523
Strategy,2150364.0,0.728,5.03,9367
Simulation,1990623.0,0.697,4.98,9427
Adventure,1961303.0,0.743,6.35,18986
Racing,1453173.0,0.708,4.51,1873
Sports,1401084.0,0.703,4.48,2248
Animation & Modeling,1359184.0,0.697,5.58,195
Design & Illustration,1147945.0,0.707,5.01,285


Databricks visualization. Run in Databricks to view.

### Interpretation du scatter Revenue vs Rating

Les genres se repartissent en quadrants distincts :
- **High revenue + high rating** : RPG, Action, Strategy — les genres les plus rentables
ET les mieux notes
- **High revenue + low rating** : Massively Multiplayer — revenus eleves malgre des
ratings inferieurs (modele F2P/monetisation agressive)
- **High rating + low revenue** : Adventure — bien note mais revenus modestes (prix plus
bas, audience niche)
- **Low revenue + low rating** : Animation & Modeling — faible attrait
commercial et critiques mitigees

Ce positionnement suggere que la qualite percue (rating) et la rentabilite ne sont pas
toujours correlees : le modele economique (F2P, prix premium, DLC) pese autant que la
satisfaction joueur.


## Conclusion -- Analyse par genre

Base : 55 690 jeux, eclatement multi-tag produisant 157 110 combinaisons jeu-genre (un jeu peut appartenir a plusieurs genres).

---

### Q7 -- Genres les plus representes

Les 5 genres les plus presents (en nombre de combinaisons jeu-genre) :

| Genre | Nombre de jeux | Part du total (157 110) |
|---|---|---|
| Indie | 39 681 | 25.3 % |
| Action | 23 759 | 15.1 % |
| Casual | 22 086 | 14.1 % |
| Adventure | 21 431 | 13.6 % |
| Strategy | 10 895 | 6.9 % |

Indie a lui seul represente un quart du catalogue. Les 4 premiers genres couvrent 68.1 % de toutes les combinaisons.

---

### Q8 -- Ratio positif/negatif par genre

Deux metriques utilisees (tous jeux confondus, filtre : total_reviews > 0, minimum 300 jeux par genre) :

- **rating_ratio** = positive_reviews / total_reviews (couverture : 99.7 % des jeux)
- **pos_neg_ratio** = positive_reviews / negative_reviews (couverture : 84.2 % -- 8 797 jeux avec zero avis negatifs exclus)

**Top 5 par avg_rating_ratio (tous jeux)** :

| Genre | avg_rating | avg_pos_neg_ratio | Jeux |
|---|---|---|---|
| Casual | 0.743 | 6.05 | 21 990 |
| Indie | 0.742 | 5.96 | 39 558 |
| Adventure | 0.739 | 6.24 | 21 397 |
| RPG | 0.731 | 5.64 | 9 520 |
| Action | 0.730 | 5.38 | 23 695 |

**Bottom 3** : Massively Multiplayer (0.629, ratio 2.70), Education (0.685, ratio 3.65) et Simulation (0.694, ratio 4.86).

Adventure obtient le meilleur pos_neg_ratio (6.24) malgre un rating_ratio legerement inferieur a Casual.

**Parmi les jeux payants uniquement** (cell 13), les tendances sont similaires : RPG (0.731), Adventure (0.739), Action (0.73). Massively Multiplayer reste en retrait (0.629).

---

### Q9 -- Publishers avec genres favoris

Top 5 publishers par volume et leurs 3 genres principaux :

| Publisher | Genre 1 | Genre 2 | Genre 3 |
|---|---|---|---|
| Big Fish Games | Casual (418) | Adventure (392) | Simulation (7) |
| 8floor | Casual (202) | Strategy (22) | Simulation (10) |
| SEGA | Action (80) | Adventure (33) | Strategy (32) |
| Strategy First | Strategy (52) | Adventure (42) | Indie (33) |
| Square Enix | Action (75) | RPG (71) | Adventure (43) |

Big Fish Games et 8floor sont quasi exclusivement Casual (respectivement 418 et 202 jeux). SEGA et Square Enix privilegient Action (80 et 75 jeux), avec Square Enix fortement positionne sur RPG (71 jeux). Strategy First est le seul du top 5 a avoir Strategy comme genre dominant (52 jeux).

---

### Q10 -- Genres les plus lucratifs

`estimated_revenue` = avg_owners x price_usd. C'est un **proxy approximatif** : les owners incluent des acquisitions gratuites (bundles, promotions), et les jeux gratuits sont exclus.

**Top 5 par revenu moyen estime (jeux payants uniquement)** :

| Genre | Revenu moyen | Revenu total | Jeux payants | Prix moyen |
|---|---|---|---|---|
| Massively Multiplayer | 8 644 545 USD | 5.93 Mds USD | 686 | 10.64 USD |
| RPG | 3 338 224 USD | 27.17 Mds USD | 8 140 | 10.59 USD |
| Action | 2 854 749 USD | 58.76 Mds USD | 20 582 | 8.92 USD |
| Strategy | 2 146 819 USD | 20.15 Mds USD | 9 386 | 9.76 USD |
| Simulation | 1 986 007 USD | 18.77 Mds USD | 9 451 | 10.42 USD |

Massively Multiplayer domine en revenu moyen (8.6M USD) mais sur un volume restreint (686 jeux). Action genere le plus gros revenu total (58.76 Mds USD) grace a son volume (20 582 jeux payants). Indie, malgre son volume massif (34 765 jeux), n'atteint que 930 435 USD de revenu moyen (prix moyen bas : 7.50 USD).

---

### Limites

- **Multi-tagging** : les 157 110 combinaisons jeu-genre proviennent de 55 690 jeux. Un meme jeu est compte dans chaque genre auquel il appartient, ce qui gonfle les volumes par genre et empeche de sommer les colonnes pour retrouver 55 690.
- **estimated_revenue est un proxy** : avg_owners x price_usd surestime les revenus reels (owners != acheteurs payants, promotions et bundles inclus, pas de donnees de ventes nettes).
- **pos_neg_ratio** : couverture limitee a 84.2 % (8 797 jeux exclus car zero avis negatifs, ce qui biaise vers les jeux ayant suffisamment de volume de reviews).
- **Seuil de 300 jeux** : les genres avec moins de 300 jeux sont exclus des analyses Q8 et Q10, ce qui elimine les niches tres petites.
- **Jeux gratuits exclus de Q10** : le revenu estime n'a pas de sens pour les jeux free-to-play dont le modele economique repose sur les microtransactions (non capturees ici).